### 1.问题
多步任务中，模型会丢失进度--重复做做过的事、跳步、跑偏。对话越长越严重：工具结果不断填满上下文，系统提示词的影响力逐渐被稀释。一个10步重构可能做完1-3步就开始即兴发挥，因为4-10步已经被挤出注意力了。
### 2.解决方法
给模型提供一个TodoManager的工具。
1. TodoManager存储带状态的项目。同一个时间只允许一个in_progress。
```python
class TodoManager:
    def update(self, items: list) -> str:
        validated, in_progress_count = [], 0
        for item in items:
            status = item.get("status", "pending")
            if status == "in_progress":
                in_progress_count += 1
            validated.append({"id": item["id"], "text": item["text"], "status": status})
        
        if in_progress_count > 1:
            raise ValueError("Only one task can be in_progress.")
        self.item = validated
        return self.render()
```
2. todo工具和其他工具一样加入 dispatch map
```python
TOOL_HANDLERS = {
    "todo": lambda **kw: TODO.update(kw["items"])
}
```
3. nag reminder: 模型连续3轮以上不调用todo时注入提醒
```python
if rounds_since_todo >= 3 and messages:
    last = messages[-1]
    if last["role"] == "user" and isinstance(last.content, list):
        last["content"].insert(0, {
            "type": "text",
            "text": "<reminder>Update your todos.</reminder>"
        })
```
同时只能有一个in_progress强制顺序聚焦。nag reminder制造问责压力。  
你不更新计划，系统就追着你问。

In [ ]:
import os
import subprocess
import json
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(base_url=os.getenv("OPENAI_BASE_URL"))

WORKDIR = Path.cwd()
SYSTEM_PROMPT = f"""You are a coding agent at {WORKDIR}.
Use the todo tool to plan multi-step tasks. Mark in_progress before starting, completed when done.
Prefer tools over prose."""

# -- TodoManager: structured state the LLM writes to --
class TodoManager:
    def __init__(self):
        self.items = []
    
    def update(self, items: list) -> str:
        if len(items) > 20:
            raise ValueError("Max 20 todos allowd")
        validated = []
        in_progress_count = 0
        for i, item in enumerate(items):
            text = str(item.get("text", "")).strip()
            status = str(item.get("status", "pending")).lower()
            item_id = str(item.get("id", str(i + 1)))
            if not text:
                raise ValueError(f"Item {item_id}: text required.")
            if status not in ("pending", "in_progress", "completed"):
                raise ValueError(f"Item {item_id}: invalid status '{status}'")
            if status == "in_progress":
                in_progress_count += 1
            validated.append({"id": item_id, "text": text, "status": status})
        if in_progress_count > 1:
            raise ValueError("Only on task can be in_progress at a time.")
        self.items = validated
        return self.render()
    
    def render(self) -> str:
        if not self.items:
            return "No todos."
        lines = []
        for item in self.items:
            marker = {"pending": "[ ]", "in_progress": "[>]", "completed": "[x]"}[item["status"]]
            lines.append(f"{marker} #{item['id']}: {item['text']}")
        done = sum(1 for t in self.items if t["status"] == "completed")
        lines.append(f"\n({done}/{len(self.items)} completed.)")
        return "\n".join(lines)

TODO = TodoManager()
    
def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

def run_bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, timeout=120)
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"

def run_read(path: str, limit: int = None) -> str:
    try:
        text = safe_path(path).read_text(encoding="utf-8", errors="ignore")
        lines = text.splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"...({len(lines) - limit} more lines)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"

def run_write(path: str, content: str) -> str:
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content)
        return f"Wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"

def run_edit(path: str, old_text: str, new_text: str) -> str:
    try:
        fp = safe_path(path)
        content = fp.read_text()
        if old_text not in content:
            return f"Error: {old_text} not found in {path}"
        fp.write_text(content.replace(old_text, new_text, 1))
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"

TOOL_HANDLERS = {
    "bash": lambda **kw: run_bash(kw["command"]),
    "read_file": lambda **kw: run_read(kw["path"]),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file": lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
    "todo": lambda **kw: TODO.update(kw["items"]),
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "bash",
            "description": "Run a shell command.",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {
                        "type": "string",
                        "description": "the shell command to execute"
                    }
                },
                "required": ["command"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read file contents.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "the path of the file",
                    },
                    "limit": {
                        "type": "int",
                        "description": "the limit of the file content.",
                    }
                },
                "required": ["path"],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "write content to file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "file path",
                    },
                    "content": {
                        "type": "string",
                        "description": "the content to write to the file.",
                    }
                },
                "required": ["path", "content"],
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "edit_file",
            "description": "Replace exact text in file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "file path",
                    },
                    "old_text": {
                        "type": "string",
                        "description": "the text to be replaced",
                    },
                    "new_text": {
                        "type": "string",
                        "description": "the text to replace the old text",
                    }
                },
                "required": ["path", "old_text", "new_text"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "todo",
            "description": "Update task list. Track progress on multi-step tasks.",
            "parameters": {
                "type": "object",
                "properties": {
                    "items": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "id": {
                                    "type": "string",
                                },
                                "text": {
                                    "type": "string",
                                },
                                "status": {
                                    "type": "string",
                                    "enum": ["pending", "in_progress", "completed"]
                                },
                            },
                            "required": [
                                "id",
                                "text",
                                "status",
                            ]
                        },
                        "required": ["items"],
                    },
                },
            }
        }
    }
]

def agent_loop(messages: list):
    rounds_since_todo = 0
    while True:
        responese = client.chat.completions.create(
            model="Pro/MiniMaxAI/MiniMax-M2.5",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        message = responese.choices[0].message
        messages.append(message)
        if not message.tool_calls:
            if message.content:
                print(message.content)
            return
        
        # 处理工具调用
        tool_results = []
        used_todo = False
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            handler = TOOL_HANDLERS.get(tool_name)
            output = handler(**json.loads(tool_call.function.arguments)) if handler else f"Unknown tool: {tool_call.function.name}"
            print(f">{tool_name}: {output[:200]}")
            tool_results.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "content": output,
            })
            if tool_name == "todo":
                used_todo = True
        rounds_since_todo = 0 if used_todo else rounds_since_todo + 1
        if rounds_since_todo >= 3:
            tool_results.insert(0, {"role": "user", "content": "<reminder>Update your todos.</reminder>"})
        
        messages.extend(tool_results)

if __name__ == "__main__":
    history = []
    while True:
        try:
            query = input("请输入: ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() in ("q", "exit", ""):
            break
        history.append({"role": "user", "content": query})
        agent_loop(history)
        response_content = history[-1].content
        if isinstance(response_content, list):
            for block in response_content:
                if hasattr(block, "text"):
                    print(block.text)
        else:
            print(response_content)
        print()



>read_file: hello
>read_file: hello
>write_file: Wrote 10 bytes to test.txt
>bash: 'rm' 不是内部或外部命令，也不是可运行的程序
或批处理文件。
>bash: (no output)
>todo: [x] #1: 读取 hello.txt 和 text.txt 的内容
[x] #2: 将内容合并写入 test.txt
[x] #3: 删除 hello.txt 和 text.txt

(3/3 completed.)


全部完成！✅

- **hello.txt** 内容：`hello`
- **text.txt** 内容：`hello`
- 已合并写入 **test.txt**：`hellohello`
- 原文件 **hello.txt** 和 **text.txt** 已删除


全部完成！✅

- **hello.txt** 内容：`hello`
- **text.txt** 内容：`hello`
- 已合并写入 **test.txt**：`hellohello`
- 原文件 **hello.txt** 和 **text.txt** 已删除

